# Fully Homomorphic Caesar Cipher (FHE Caesar Cipher)

Implementing Fully Homomorphic computing using a caesar cipher.

The Caesar Cipher itself is completely insecure due to:
- frequency analysis (the distribution of values remains the same, it's just shifted)

However, if we instead used a One Time pad for the shifts, surely Shannon's proof gives us \
information-theoretic guarantees that it's secure?

Ok so we get addition for free, you just need to know the compute graph and reverse the key accumulation \
(thats easy, just subtract the keys which only we know).

And we can achieve multiplication using one-hot encoding of input values, dot product that over the possible \
outputs (which we can chain together which gives us unary, binary or n-ary functions).

Then we need to somehow convert back to integer form if we want to perform addition again, or we just use lookup \
tables for that as well?

The lookup tables can be implemented very efficiently via matmul. And if we can decompose the representation format \
into something which isn't literally just one-hot encoding and lookups then I reckon we can save on space as well.

This might be an effective way of having information-theoreticly secure remote neural network execution.

## Addition

In [1]:
def encrypt(ll, shift, mod):
    return [(i+shift) % mod for i in ll]

def decrypt(ll, shift, mod):
    return [(i-shift) % mod for i in ll]

def add(a, b, mod):
    assert len(a) == len(b)
    return [(a[i] + b[i]) % mod for i in range(len(a))]

ll = [5000]
k = 100
mod = 10_000_000
a = encrypt(ll, k, mod)
b = encrypt(ll, k, mod)
s = add(a, b, mod)

a, b, s, decrypt(s, k*2, mod)

([5100], [5100], [10200], [10000])

## Multiplication

In [3]:
# # 1x1 to 2x2
# table = [
#     [1, 1, 1],
#     [1, 2, 2],
#     [2, 1, 2],
#     [2, 2, 4]
# ]

# print(table)

# # preshift table with key?
# for i in range(len(table)): table[i] = encrypt(table[i], k, mod)

# print("tb:", table)

# def multiply(a, b, table):
#     assert len(a) == len(b)
#     def product(b, c, table):
#         print(b, c, table)
#         # assuming table has product
#         return [row[2] for row in table if row[0] == b and row[1] == c][0]
    
#     return sum([product(a[i], b[i], table) for i in range(len(a))])

# multiply(a, b, table) - (k * len(a))

## One-Hot Lookup

In [ ]:
import numpy as np

table = [
    [1, 1, 1],
    [1, 2, 2],
    [2, 1, 2],
    [2, 2, 4]
]

def gen_onehot_table(entries):
    table = []
    flatten = lambda ll: [i for x in ll for i in x]
    items = flatten(entries)
    items = set(items)
    l = len(items)
    items = list(items)
    items.sort()
    mapping = {idx:itm for itm, idx in enumerate(items)}
    print(mapping, items)

    def one_hot(v, l, d=[0, 1]):
        print(v)
        l = [d[0] if i != v-1 else d[1] for i in range(l)][::-1]
        return l
    
    for r in entries:
        a, b, c = r
        new_a = one_hot(mapping[a]+1, l)
        new_b = one_hot(mapping[b]+1, l)
        new_c = one_hot(mapping[c]+1, l)
        new_c[new_c.index(1)] = c
        table.append([new_a, new_b, new_c])

    return table

a, b = [0, 0, 1], [0, 0, 1]

mult_table = gen_onehot_table(table)

def match(x, y, tbl):
    for r in tbl:
        a, b, c = r
        i = np.dot(a, x)
        j = np.dot(b, y)
        print("ij:", a, b, c, i, j)

print(mult_table)

match(a, b, mult_table)

{1: 0, 2: 1, 4: 2} [1, 2, 4]
1
1
1
1
2
2
2
1
2
2
2
3
[[[0, 0, 1], [0, 0, 1], [0, 0, 1]], [[0, 0, 1], [0, 1, 0], [0, 2, 0]], [[0, 1, 0], [0, 0, 1], [0, 2, 0]], [[0, 1, 0], [0, 1, 0], [4, 0, 0]]]
ij: [0, 0, 1] [0, 0, 1] [0, 0, 1] 1 1
ij: [0, 0, 1] [0, 1, 0] [0, 2, 0] 1 0
ij: [0, 1, 0] [0, 0, 1] [0, 2, 0] 0 1
ij: [0, 1, 0] [0, 1, 0] [4, 0, 0] 0 0


## Multi One-Hot Lookup

In [22]:
import random

mod = 10_000_007  # prime modulus
domain = 4        # values: 0, 1, 2, 3

# this is known ahead of time (and is plaintext)
# multiplication table: T[i][j] = i * j
T = [[i * j for j in range(domain)] for i in range(domain)]

for r in T:
    print(r)

def one_hot(val, size):
    return [1 if i == val else 0 for i in range(size)]

def encrypt_one_hot(oh, mod):
    # keys = [random.randint(0, mod - 1) for _ in oh]
    keys = [1 for _ in oh]
    enc = [(v + k) % mod for v, k in zip(oh, keys)]
    return enc, keys

# === TEST ALL PAIRS ===
for a in range(domain):
    for b in range(domain):
        # encryptd vals a and b (only alice can decrypt this)
        a_oh = one_hot(a, domain)
        b_oh = one_hot(b, domain)
        E_a, K_a = encrypt_one_hot(a_oh, mod)
        E_b, K_b = encrypt_one_hot(b_oh, mod)

        # --- SERVER (knows T, sees E_a and E_b, knows nothing else) ---

        # Step 1: select row using encrypted one-hot a
        row = [sum(T[i][j] * E_a[i] for i in range(domain)) % mod
               for j in range(domain)]

        if a == 2 and b == 2:
            print("row:", row, "\n", "E_a:", E_a, "\n", "K_a:", K_a, "\n", "E_b:", E_b, "\n", "K_b:", K_b)

        # Step 2: use row values as constants against encrypted one-hot b
        result = sum(row[j] * E_b[j] for j in range(domain)) % mod

        if a == 2 and b == 2:
            print("result:", result)
        
        # --- ALICE (knows keys, table, her plaintexts) ---

        # we need the intermediate for the correction?
        # correction1: remove key_b contribution
        correction1 = sum(row[j] * K_b[j] for j in range(domain)) % mod

        # correction2: remove key_a noise that leaked through b's selection
        noise = [sum(T[i][j] * K_a[i] for i in range(domain)) % mod
                 for j in range(domain)]
        correction2 = sum(noise[j] * b_oh[j] for j in range(domain)) % mod

        decrypted = (result - correction1 - correction2) % mod
        expected = a * b
        if a == 2 and b == 2:
            print("result, correction1, correction2, expected:", result, correction1, correction2, expected)
        status = "✓" if decrypted == expected else "✗"
        if a == 2 and b == 2:
            print(f"  {a} × {b} = {decrypted:2d}  (expected {expected:2d}) {status}")

[0, 0, 0, 0]
[0, 1, 2, 3]
[0, 2, 4, 6]
[0, 3, 6, 9]
row: [0, 8, 16, 24] 
 E_a: [1, 1, 2, 1] 
 K_a: [1, 1, 1, 1] 
 E_b: [1, 1, 2, 1] 
 K_b: [1, 1, 1, 1]
result: 64
result, correction1, correction2, expected: 64 48 12 4
  2 × 2 =  4  (expected  4) ✓


In [92]:
d = 4
v = [0, 0, 1, 0]
t = [99, 3, 3, 4]
s = [1, 2, 3, 4]

new_v = [1, 0, 0, 0]

def go(v, t, s, rev=-1):
    rev = sum([t[i] * s[i] for i in range(d)])
    v = [v[i] + s[i] for i in range(d)]
    p = [v[i] * t[i] for i in range(d)]
    r = sum(p)
    print(v, t, s, p, r, r - rev)
    return new_v, r, rev

new_v, r, rev = go(v, t, s)
print(new_v, r, rev)

[1, 2, 4, 4] [99, 3, 3, 4] [1, 2, 3, 4] [99, 6, 12, 16] 133 3
[1, 0, 0, 0] 133 130


In [ ]:
1

3 2
2 1

3 - 2 = 1 (0) + (1+1) = 2